In [ ]:
# !pip install transformers, AutoTokenizer, torch

In [8]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch
import numpy as np

In [9]:
MODEL_NAME = "gpt2"
DATASET_NAME = "wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "train"

BLOCK_SIZE = 128
STRIDE = 64
BATCH_SIZE = 8
SEED = 42

In [10]:
print(
    f"Model: {MODEL_NAME}, Dataset: {DATASET_NAME}/{DATASET_CONFIG}, "
    f"split={SPLIT}, block_size={BLOCK_SIZE}, stride={STRIDE}, batch_size={BATCH_SIZE}"
)

Model: gpt2, Dataset: wikitext/wikitext-2-raw-v1, split=train, block_size=128, stride=64, batch_size=8


In [11]:
raw_ds = load_dataset(DATASET_NAME, DATASET_CONFIG, split=SPLIT)
print(f"Raw dataset size: {len(raw_ds)}")

splits = raw_ds.train_test_split(test_size=0.05, seed=SEED)
train_ds = splits["train"]
val_ds = splits["test"]

print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}")
print("Example raw row:", train_ds[0])

Raw dataset size: 36718
Train size: 34882, Val size: 1836
Example raw row: {'text': " Although Dover finished in eighth place in their first season in the Conference , the following season saw the club struggling against relegation , and Kinnear was dismissed due to a combination of the team 's poor performances and his own personal problems . John Ryan was appointed as the club 's new manager , but his reign was a short one and he was dismissed when the club lost seven of its first eight matches in the 1995 – 96 season . The club then appointed former England international Peter Taylor as manager , but he was unable to steer the team away from the foot of the table , and Dover held onto their place in the Conference only because Northern Premier League runners @-@ up Boston United failed to submit their application for promotion before the required deadline . \n"}


In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer pad token id: {tokenizer.pad_token_id}")

Tokenizer pad token id: 50256


In [13]:
def tokenize_batch(examples):
    """
    Tokenize a batch of raw text lines.
    We don't truncate here; we handle long sequences when grouping.
    """
    return tokenizer(
        examples["text"],
        add_special_tokens=False,
        truncation=False
    )

In [14]:
tokenized_train = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_ds.column_names,
)
tokenized_val = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=val_ds.column_names,
)

Map:   0%|          | 0/34882 [00:00<?, ? examples/s]

Map:   0%|          | 0/1836 [00:00<?, ? examples/s]

In [15]:
print("Tokenized train example (first 20 token ids):", tokenized_train[0]["input_ids"][:20])

Tokenized train example (first 20 token ids): [4900, 46578, 5201, 287, 16974, 1295, 287, 511, 717, 1622, 287, 262, 8785, 837, 262, 1708, 1622, 2497, 262, 3430]


In [16]:
train_lengths = [len(x) for x in tokenized_train["input_ids"]]
print("\nToken length stats on train split:")
print("  mean:", float(np.mean(train_lengths)))
print("  median:", float(np.median(train_lengths)))
print("  95th percentile:", float(np.percentile(train_lengths, 95)))
print("  max:", int(np.max(train_lengths)))


Token length stats on train split:
  mean: 65.20775758270742
  median: 11.0
  95th percentile: 246.0
  max: 841


In [17]:
def group_texts_sliding(examples, block_size: int = BLOCK_SIZE, stride: int = STRIDE):
    """
    Concatenate all input_ids in the batch, then create overlapping blocks
    using a sliding window of size `block_size` with step `stride`.
    """
    # Flatten list-of-lists into a single list
    all_ids = []
    for ids in examples["input_ids"]:
        all_ids.extend(ids)

    result_input_ids = []
    for start in range(0, len(all_ids) - block_size + 1, stride):
        chunk = all_ids[start : start + block_size]
        result_input_ids.append(chunk)

    # labels are just a copy of input_ids for causal LM
    result = {
        "input_ids": result_input_ids,
        "labels": [ids.copy() for ids in result_input_ids],
    }
    return result

In [25]:
lm_train = tokenized_train.map(
    group_texts_sliding,
    batched=True,
    remove_columns=tokenized_train.column_names
)

lm_val = tokenized_val.map(
    group_texts_sliding,
    batched=True,
    remove_columns=tokenized_val.column_names
)


Map:   0%|          | 0/34882 [00:00<?, ? examples/s]

Map:   0%|          | 0/1836 [00:00<?, ? examples/s]

In [26]:
print(f"\nLM train sequences (sliding): {len(lm_train)}")
print(f"LM val sequences   (sliding): {len(lm_val)}")
print("Example LM train seq length:", len(lm_train[0]["input_ids"]))


LM train sequences (sliding): 35488
LM val sequences   (sliding): 1830
Example LM train seq length: 128


In [27]:
def collate_fn(batch):
    """
    Collate a list of examples into a batch for causal LM:
    - Pad input_ids to max length in batch.
    - Copy to labels.
    - Set labels to -100 at padding positions (ignored in loss).
    - Build attention_mask.
    """
    input_ids = [torch.tensor(ex["input_ids"], dtype=torch.long) for ex in batch]
    labels    = [torch.tensor(ex["labels"],    dtype=torch.long) for ex in batch]

    # Pad to the longest sequence in the batch
    input_ids_padded = torch.nn.utils.rnn.pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=tokenizer.pad_token_id,
    )
    labels_padded = torch.nn.utils.rnn.pad_sequence(
        labels,
        batch_first=True,
        padding_value=tokenizer.pad_token_id,
    )

    # Mask out padding in labels
    pad_mask = input_ids_padded.eq(tokenizer.pad_token_id)
    labels_padded = labels_padded.masked_fill(pad_mask, -100)

    attention_mask = input_ids_padded.ne(tokenizer.pad_token_id).long()

    return {
        "input_ids": input_ids_padded,
        "labels": labels_padded,
        "attention_mask": attention_mask,
    }

In [28]:
train_loader = DataLoader(
    lm_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    lm_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

In [29]:
batch = next(iter(train_loader))
print("\nBatch shapes:")
print("  input_ids:      ", batch["input_ids"].shape)
print("  labels:         ", batch["labels"].shape)
print("  attention_mask: ", batch["attention_mask"].shape)


Batch shapes:
  input_ids:       torch.Size([8, 128])
  labels:          torch.Size([8, 128])
  attention_mask:  torch.Size([8, 128])


In [32]:
num_pad_labels = (batch["labels"] == -100).sum().item()
print("  # of -100 label positions (should match padding):", num_pad_labels)

  # of -100 label positions (should match padding): 0


In [33]:
example_ids = lm_train[0]["input_ids"]
decoded = tokenizer.decode(example_ids)
print("\nDecoded first training sequence (truncated):\n")
print(decoded[:500], "...")


Decoded first training sequence (truncated):

 Although Dover finished in eighth place in their first season in the Conference , the following season saw the club struggling against relegation , and Kinnear was dismissed due to a combination of the team 's poor performances and his own personal problems . John Ryan was appointed as the club 's new manager , but his reign was a short one and he was dismissed when the club lost seven of its first eight matches in the 1995 – 96 season . The club then appointed former England international Pete ...
